# Where is the grid refined? The AMR levels of a galaxy, on one page

An adaptive-mesh simulation spends its cells where the gas needs them. This recipe makes that
visible: how many cells, how much volume and how much mass sit on each refinement level, which
level holds the gas along every line of sight, how thick the refined zone is seen from the side,
the exact level in the mid-plane, the cell edges themselves drawn over the gas, and finally the
grid in three dimensions, on the faces of a small cube.

It answers practical questions before any science: is the region I care about resolved? Where does
the resolution drop? Does a feature in my map sit on a refinement boundary?

**To run it on your own simulation, change two things in the next code cell**: the path and the
output number. Any RAMSES output or converted MERA file with an AMR grid works.

> This is a gallery recipe. It is not part of Mera's test suite, and its outputs are not stored in
> the notebook: run it and it will produce them. It was written on the AVALON galaxy simulation,
> which is not public. No simulation at hand? The next cell has a line for one of Mera's public
> test runs.

**What you need:** a snapshot with gas on an AMR grid, and CairoMakie (it writes the movie
itself, through the FFMPEG it ships with). The run shown here reads the
AVALON galaxy at level 13, from a working copy cut to the disc region (±16 kpc across, ±6 kpc
high) that keeps only the density: 154.5 million cells, about 10 GB of memory and six minutes. The
full snapshot of the same run needs about 20 GB; a level-12 run of 51.5 million cells, about 5 GB.

![The leaf cells of the adaptive grid on three faces of a cube, coloured by level](media/grid_3d.png)

| | |
|---|---|
| **Author** | Manuel Behrendt |
| **Contact** | [github.com/ManuelBehrendt](https://github.com/ManuelBehrendt) |
| **Reads** | RAMSES |
| **Mera** | 1.8.0, Julia 1.12.7 |
| **Provenance** | `Mera v1.8.0 \| L13/output_00390 \| 578.6 Myr \| L=48.0 ndim=3 lmin=6 lmax=13` |
| **Status** | contributed 2026-10-20 |

## The environment

This recipe carries its own `Project.toml` and `Manifest.toml`. Running the cell below uses exactly
the package versions it was written against. See
[Reproducibility](https://manuelbehrendt.github.io/Mera.jl/stable/reproducibility/).

In [ ]:
using Pkg
Pkg.activate(@__DIR__)
Pkg.instantiate()

## 1. The two lines you change

In [ ]:
using Mera, Printf
using CairoMakie
CairoMakie.activate!(type = "png")

# ---- change these two ----------------------------------------------------------------
SIM    = "/path/to/your/simulation"   # a RAMSES output folder, or a MERA-file folder
OUTPUT = 390                          # the snapshot number
# --------------------------------------------------------------------------------------
# No simulation at hand? A public test run works too. Its blast sits in the box corner,
# so look there, and its box is 0.5 code units wide:
#   SIM, OUTPUT = download_testdata("sedov3d_amr"), 7
#   CENTER, RU, FRAME, HEIGHT = [0., 0., 0.], :standard, 0.25, 0.25
#   ZOOM = (x = (0.05, 0.15), y = (0.0, 0.1)); CUBE = (center = (0.1, 0.05, 0.05), half = 0.05)

CENTER = [:bc]                        # where to look: the box centre
RU     = :kpc                         # the unit of FRAME and ZOOM
FRAME  = 16.0                         # half-width of the maps
ZOOM   = (x = (-10.0, -9.0), y = (-5.8, -4.8)) # a small window across a spiral arm
HEIGHT = 6.0                         # half-height of the side views
CUBE   = (center = (-9.5, -5.45, 0.0), half = 0.25)   # the 3-D view: a cube on the same arm
NFRAMES = 180                        # the rotation movie: 6 s at 30 fps; try 12 first

## 2. Read the gas

Only the gas is needed: the grid is a property of the hydro cells. `loadall` reads a RAMSES output
and a MERA file with the same call.

**Cost.** A MERA file is read as a whole before any selection, so the full gas table of this file,
154.5 million cells with only the density, sits in memory: about 6 GB. Cutting a working copy once
(column and window, then `savedata`) is what made the level-13 run fit on a laptop. For a RAMSES output you can pass `lmax` or a window
(`xrange=…, center=…`) to `loadall` to read less.

In [ ]:
t_read = @elapsed (; hydro, info) = loadall(SIM, OUTPUT; components = (:hydro,), verbose = false)

scale = getfield(info.scale, RU)      # code length -> RU
@printf("%d cells on levels %d to %d, read in %.0f s\n",
        length(hydro.data), hydro.lmin, hydro.lmax, t_read)

## 3. Counting first: cells, volume and mass per level

Before any map, a table. Each level halves the cell size, so one level up means eight times as many
cells for the same volume. The three columns answer three different questions: where the *work*
goes (cells), where the *space* is (volume), and where the *gas* is (mass).

In [ ]:
lv   = getvar(hydro, :level)
vol  = getvar(hydro, :volume, :kpc3)
mass = getvar(hydro, :mass, :Msol)

levels = hydro.lmin:hydro.lmax
cellpc = [hydro.boxlen * info.scale.pc / 2^l for l in levels]
ncell  = [count(==(l), lv) for l in levels]
vfrac  = [sum(vol[lv .== l]) for l in levels]  ./ sum(vol)
mfrac  = [sum(mass[lv .== l]) for l in levels] ./ sum(mass)

println("level  cell size    cells      volume     mass")
for (k, l) in enumerate(levels)
    @printf("%4d  %8.1f pc  %7.3f %%  %7.3f %%  %7.3f %%\n",
            l, cellpc[k], 100ncell[k] / length(lv), 100vfrac[k], 100mfrac[k])
end
vol = mass = nothing                  # keep only what the next steps need


# ── figure code from here: no new Mera concepts ──────────────────────────────────────
pal   = cgrad(:viridis, length(levels); categorical = true)   # one colour per level, everywhere
fig_c = Figure(size = (1100, 330))
for (k, (vals, ttl)) in enumerate(((ncell ./ length(lv), "share of cells"),
                                   (vfrac, "share of volume"),
                                   (mfrac, "share of mass")))
    ax = Axis(fig_c[1, k], title = ttl, xlabel = "level", yscale = log10,
              xticks = collect(levels), ylabel = k == 1 ? "fraction" : "")
    barplot!(ax, collect(levels), max.(vals, 1e-7); color = collect(levels), colormap = pal,
             colorrange = (first(levels) - 0.5, last(levels) + 0.5), fillto = 1e-7)
    ylims!(ax, 1e-6, 1.5)
end
mkpath("media"); save("media/grid_levels_counts.png", fig_c)
fig_c

![Cells, volume and mass per level](media/grid_levels_counts.png)

On this galaxy the three panels tell three different stories. Levels 8 and 9 (cells of 190 and
94 pc) fill 94 % of the volume but hold 6 % of the gas. The finest level, 13 with cells of 6 pc,
fills 0.14 % of the volume, yet holds 57 % of all cells and 57 % of the mass. That is adaptive
refinement doing its job: the cells go where the mass is. Levels 6 and 7 are empty here because
this file holds only the disc region, where the grid is at least level 8.

## 4. Which level holds the gas, seen from above

A projection can average any quantity along the line of sight, and the level is a quantity like any
other. Two weightings answer two different questions:

- **weighted by mass**: the level on which the *gas* in this column sits;
- **weighted by volume**: the level that fills the *space* in this column.

The result is a mean, so it is not a whole number: a column that crosses fine disc cells and coarse
halo cells lands in between. The mass-weighted map uses one colour per level, rounding the mean to
the nearest level, which shows where each level dominates. The volume-weighted map uses a
continuous scale instead, which shows the real gradient. The surface density on the left is the
context.

In [ ]:
view = (direction = :z, center = CENTER, range_unit = RU,
        xrange = [-FRAME, FRAME], yrange = [-FRAME, FRAME], verbose = false, show_progress = false)

p_mass = projection(hydro, [:sd, :level], [:Msol_pc2, :standard]; weighting = [:mass], view...)
p_vol  = projection(hydro, :level; weighting = [:volume], view...)


# ── figure code from here: no new Mera concepts ──────────────────────────────────────
axes_of(p, m) = (range(p.cextent[1], p.cextent[2]; length = size(m, 1)) .* scale,
                 range(p.cextent[3], p.cextent[4]; length = size(m, 2)) .* scale)
LCR    = (first(levels) - 0.5, last(levels) + 0.5)
lticks = (collect(levels), [@sprintf("%d  (%.0f pc)", l, c) for (l, c) in zip(levels, cellpc)])

fig_m = Figure(size = (1250, 430))
sd = p_mass.maps[:sd]
ax1 = Axis(fig_m[1, 1], title = "surface density", aspect = DataAspect())
hm1 = heatmap!(ax1, axes_of(p_mass, sd)..., log10.(max.(sd, 1e-3)); colormap = :inferno,
               colorrange = (-1.5, 2.5))
ax2 = Axis(fig_m[1, 2], title = "mean level, weighted by mass", aspect = DataAspect())
heatmap!(ax2, axes_of(p_mass, sd)..., p_mass.maps[:level]; colormap = pal, colorrange = LCR)
ax3 = Axis(fig_m[1, 3], title = "mean level, weighted by volume", aspect = DataAspect())
lvol = p_vol.maps[:level]
VCR  = extrema(filter(v -> isfinite(v) && v > 0, lvol))       # its own range; 0 = no data
hm3  = heatmap!(ax3, axes_of(p_vol, lvol)..., lvol; colormap = :viridis, colorrange = VCR)
for ax in (ax1, ax2, ax3); ax.xlabel = "x [$RU]"; end
ax1.ylabel = "y [$RU]"; hideydecorations!(ax2; grid = false); hideydecorations!(ax3; grid = false)
Colorbar(fig_m[2, 1], hm1; vertical = false, label = "log₁₀ Σ  [Msol pc⁻²]", flipaxis = false)
Colorbar(fig_m[2, 2]; colormap = pal, colorrange = LCR, vertical = false, ticks = collect(levels),
         label = "refinement level", flipaxis = false)
Colorbar(fig_m[2, 3], hm3; vertical = false, label = "mean level (continuous)", flipaxis = false)
save("media/grid_levels_maps.png", fig_m)
fig_m

![Surface density and the mean level along each line of sight](media/grid_levels_maps.png)

Weighted by mass, the map follows the gas: the spiral arms and clumps sit on level 13, the gas
between them on levels 11 and 12, the outer disc on level 10, and the space beyond the disc on
level 8. Weighted by volume, the picture is smooth: most of the volume in each column lies above
and below the disc, where cells are large, so the mean rises only slowly toward the centre, where
the refined zone is thickest (step 5).

## 5. Seen from the side: how thick is the refined zone?

From above, the refinement follows the disc. The side view says how far it reaches above and below
the mid-plane, and whether the refined zone is a thin cylinder or rather a sphere. The same two
projections, now along x (`direction = :x`), and a slice through the vertical plane
(`los = [1, 0, 0]`), which shows the exact levels there.

In [ ]:
side = (direction = :x, center = CENTER, range_unit = RU, xrange = [-FRAME, FRAME],
        yrange = [-FRAME, FRAME], zrange = [-HEIGHT, HEIGHT], verbose = false, show_progress = false)
s_mass = projection(hydro, :level; weighting = [:mass],   side...)
s_vol  = projection(hydro, :level; weighting = [:volume], side...)
s_sl   = slice(hydro, :level; los = [1, 0, 0], center = CENTER, range_unit = RU,
               xrange = [-FRAME, FRAME], yrange = [-HEIGHT, HEIGHT], pxsize = [FRAME / 400, RU],
               verbose = false)


# ── figure code from here: no new Mera concepts ──────────────────────────────────────
fig_e = Figure(size = (1100, 900))
ax1 = Axis(fig_e[1, 1], title = "mean level along x, weighted by mass", aspect = DataAspect(),
           ylabel = "z [$RU]")
heatmap!(ax1, axes_of(s_mass, s_mass.maps[:level])..., s_mass.maps[:level]; colormap = pal,
         colorrange = LCR)
ax2 = Axis(fig_e[2, 1], title = "mean level along x, weighted by volume (continuous)",
           aspect = DataAspect(), ylabel = "z [$RU]")
sv  = s_vol.maps[:level]
hm2 = heatmap!(ax2, axes_of(s_vol, sv)..., sv; colormap = :viridis,
               colorrange = extrema(filter(v -> isfinite(v) && v > 0, sv)))
xs = range(s_sl.extent[1], s_sl.extent[2]; length = size(s_sl.map, 1)) .* scale
zs = range(s_sl.extent[3], s_sl.extent[4]; length = size(s_sl.map, 2)) .* scale
ax3 = Axis(fig_e[3, 1], title = "exact level in the plane x = centre", aspect = DataAspect(),
           xlabel = "y [$RU]", ylabel = "z [$RU]")
heatmap!(ax3, xs, zs, s_sl.map; colormap = pal, colorrange = LCR)
Colorbar(fig_e[1, 2]; colormap = pal, colorrange = LCR, ticks = collect(levels), label = "level")
Colorbar(fig_e[2, 2], hm2; label = "mean level")
Colorbar(fig_e[3, 2]; colormap = pal, colorrange = LCR, ticks = collect(levels), label = "level")
save("media/grid_levels_side.png", fig_e)
fig_e

![The refinement levels seen from the side](media/grid_levels_side.png)

From the side, the refined zone is a thin disc with a round core. Levels 12 and 13 stay within
about half a kiloparsec of the mid-plane over the whole disc. Around the centre, levels 10 and 11
swell into a spheroid that reaches about 2 kpc (level 11) and 4 kpc (level 10) above and below the
plane, much more than the disc is thick. So the refined zone is neither a pure cylinder nor a pure
sphere: a thin disc plus a spheroid around the centre. The mass-weighted view also shows refined
gas lifted up to about 2 kpc above the disc. The views stop at ±6 kpc because this file does.

## 6. The exact level, in one plane

A mean blurs the boundaries. A **slice** does not: in its nearest-cell mode it paints each pixel
with the one cell that contains it, so every pixel shows a whole level. Giving a line of sight
(`los=[0,0,1]`, looking down the z axis) and a pixel size selects that mode. The mid-plane, through
`CENTER`, is where the disc is resolved best.

`.cellsize` comes with it: the size of the cell that won each pixel. Comparing it with the pixel
size tells whether the map is finer than the data.

In [ ]:
sl = slice(hydro, :level; los = [0, 0, 1], center = CENTER, range_unit = RU,
           xrange = [-FRAME, FRAME], yrange = [-FRAME, FRAME], pxsize = [FRAME / 400, RU],
           verbose = false)
px_code = (sl.extent[2] - sl.extent[1]) / size(sl.map, 1)
@printf("%d x %d pixels of %.1f pc; finest cell %.1f pc\n", size(sl.map)...,
        px_code * info.scale.pc, minimum(cellpc))


# ── figure code from here: no new Mera concepts ──────────────────────────────────────
xs = range(sl.extent[1], sl.extent[2]; length = size(sl.map, 1)) .* scale
ys = range(sl.extent[3], sl.extent[4]; length = size(sl.map, 2)) .* scale
fig_s = Figure(size = (640, 600))
ax = Axis(fig_s[1, 1], title = "level in the mid-plane (exact, one cell per pixel)",
          aspect = DataAspect(), xlabel = "x [$RU]", ylabel = "y [$RU]")
heatmap!(ax, xs, ys, sl.map; colormap = pal, colorrange = LCR)
lines!(ax, [ZOOM.x[1], ZOOM.x[2], ZOOM.x[2], ZOOM.x[1], ZOOM.x[1]],
       [ZOOM.y[1], ZOOM.y[1], ZOOM.y[2], ZOOM.y[2], ZOOM.y[1]]; color = :white, linewidth = 1.5)
text!(ax, ZOOM.x[1], ZOOM.y[2]; text = "step 7", color = :white, align = (:left, :bottom),
      fontsize = 12)
Colorbar(fig_s[1, 2]; colormap = pal, colorrange = LCR, ticks = lticks, label = "level  (cell size)")
save("media/grid_levels_slice.png", fig_s)
fig_s

![The exact level in the mid-plane](media/grid_levels_slice.png)

In the plane of the disc the picture is sharp: the arms and clumps are level 13, the gas between
them steps down through 12, 11 and 10, the outer disc is level 9, and only the corners of the frame
reach level 8. The white box is the window of step 7.

## 7. The cell edges themselves

`gridoverlay` returns the edges of all cells of one level as line segments, ready to draw over a
map. Two choices make the picture readable:

- **A thin slab around the mid-plane** (`zrange`). The edges of a level are its footprint along the
  line of sight. Through the whole disc that footprint is a dense mesh; in a slab one finest cell
  thick it is exactly the grid in that plane.
- **A small window** (`xrange`, `yrange`). The number of segments grows four times per level, so
  the finest levels over the whole galaxy would be millions of lines.

Each level gets its colour from the same palette as the maps above. Underneath, the gas density in
the same plane, from the same nearest-cell slice.

Only **leaf cells** are drawn, the cells that are not refined any further. Mera's gas table holds
only those, never their parent cells, so each level appears only where it is the finest: the coarse
grid stops at the edge of a refined patch instead of running on underneath it.

In [ ]:
cs_min = hydro.boxlen / 2^hydro.lmax * scale       # the finest cell, in RU
slab   = [-cs_min / 2, cs_min / 2]                 # one finest cell thick, around the plane
win    = (center = CENTER, range_unit = RU, xrange = collect(ZOOM.x), yrange = collect(ZOOM.y),
          zrange = slab)

edges = Dict(l => gridoverlay(hydro; level = l, direction = :z, unit = RU, win...)
             for l in levels)
for l in levels
    n = length(edges[l].segments)
    n > 0 && @printf("level %2d: %7d edges\n", l, n)
end

rho = slice(hydro, :rho, :nH; los = [0, 0, 1], center = CENTER, range_unit = RU,
            xrange = collect(ZOOM.x), yrange = collect(ZOOM.y), pxsize = [cs_min / 2, RU],
            verbose = false)

# gridoverlay works in box coordinates, the slice around CENTER: shift the edges to the slice
c0 = Mera.prepboxcenter(info, RU, CENTER)          # CENTER in RU, measured from the box corner


# ── figure code from here: no new Mera concepts ──────────────────────────────────────
# a windowed slice is centred on its window, so add the window's middle back
xs = range(rho.extent[1], rho.extent[2]; length = size(rho.map, 1)) .* scale .+ sum(ZOOM.x) / 2
ys = range(rho.extent[3], rho.extent[4]; length = size(rho.map, 2)) .* scale .+ sum(ZOOM.y) / 2
fig_z = Figure(size = (760, 640))
ax = Axis(fig_z[1, 1], title = "cell edges in the mid-plane, coloured by level",
          aspect = DataAspect(), xlabel = "x [$RU]", ylabel = "y [$RU]")
hm = heatmap!(ax, xs, ys, log10.(max.(rho.map, 1e-4)); colormap = :grays, colorrange = (-1.5, 2.0))
for (k, l) in enumerate(levels)
    s = edges[l].segments
    isempty(s) && continue
    pts = Point2f[]
    for (x1, y1, x2, y2) in s
        push!(pts, Point2f(x1 - c0[1], y1 - c0[2]), Point2f(x2 - c0[1], y2 - c0[2]))
    end
    linesegments!(ax, pts; color = (pal[k], 0.9), linewidth = l == last(levels) ? 0.5 : 1.0)
end
limits!(ax, ZOOM.x..., ZOOM.y...)
Colorbar(fig_z[1, 2], hm; label = "log₁₀ n_H  [cm⁻³]")
Colorbar(fig_z[1, 3]; colormap = pal, colorrange = LCR, ticks = lticks, label = "level  (cell size)")
save("media/grid_zoom_edges.png", fig_z)
fig_z

![The cell edges across the edge of the disc](media/grid_zoom_edges.png)

Every refinement step is visible across one spiral arm: level-13 cells (yellow) fill the densest
gas, level-12 cells surround them, and a rim of level-11 cells separates them from the level-10 gas
between the arms. That rim is not a
choice of this recipe: RAMSES lets neighbouring cells differ by at most one level, so a refined
region always steps down one level at a time.

## 8. The grid in three dimensions

A map flattens one direction away. To see the grid as the 3-D structure it is, take a small cube
on the same arm, 500 pc wide and reaching 250 pc above and below the mid-plane, and draw the edges
of **every** leaf cell inside it, transparent, so that you can look through the whole cube.

Mera has no 3-D grid function, and none is needed: the edges come straight from the cell indices.
A cell of level `l` spans `2^(lmax - l)` cells of the finest level, so in units of the finest cell
every corner is a whole number. An edge shared by neighbouring cells then has the same key, and a
`Set` keeps it once. Without that, every inner edge would be drawn up to four times.

In [ ]:
C3 = c0 .+ collect(CUBE.center)          # the cube's centre, from the box corner, in RU
h  = CUBE.half
cube = subregion(hydro, cuboid(xrange = [-h, h], yrange = [-h, h], zrange = [-h, h],
                               center = C3, range_unit = RU); split = false, verbose = false)

function cell_edges(obj, l)              # the 12 edges of each cell of level l, each edge once
    lv = getvar(obj, :level); sel = lv .== l
    cx = getvar(obj, :cx)[sel]; cy = getvar(obj, :cy)[sel]; cz = getvar(obj, :cz)[sel]
    s  = 2^(obj.lmax - l)                # this level's cell size, in finest cells
    E  = Set{NTuple{6,Int}}()
    for k in eachindex(cx)
        x0, y0, z0 = (cx[k] - 1) * s, (cy[k] - 1) * s, (cz[k] - 1) * s
        for a in (0, s), b in (0, s)
            push!(E, (x0, y0 + a, z0 + b, x0 + s, y0 + a, z0 + b))    # along x
            push!(E, (x0 + a, y0, z0 + b, x0 + a, y0 + s, z0 + b))    # along y
            push!(E, (x0 + a, y0 + b, z0, x0 + a, y0 + b, z0 + s))    # along z
        end
    end
    return E
end

lv_cube = getvar(cube, :level)
edges3  = Dict(l => cell_edges(cube, l) for l in levels)
for l in levels
    n = count(==(l), lv_cube)
    n > 0 && @printf("level %2d: %6d cells, %7d edges\n", l, n, length(edges3[l]))
end


# ── figure code from here: no new Mera concepts ──────────────────────────────────────
fin  = hydro.boxlen * scale / 2^hydro.lmax                    # the finest cell, in RU
to3(i, j, k) = Point3f(i * fin - C3[1], j * fin - C3[2], k * fin - C3[3])
used = [l for l in levels if !isempty(edges3[l])]
# few coarse cells drawn strongly, many fine ones faintly, so the inside stays visible
alpha(l) = 0.6 * 0.6^(l - first(used))

fig3 = Figure(size = (1000, 1000), backgroundcolor = :black)
ax = Axis3(fig3[1, 1]; aspect = :data, perspectiveness = 0.4, azimuth = 1.275π, elevation = π / 7,
           backgroundcolor = :black)
hidedecorations!(ax); hidespines!(ax)
drawn = Any[]                                      # every drawn object, to turn them together
for (k, l) in enumerate(levels)
    l in used || continue
    P = Point3f[]; sizehint!(P, 2length(edges3[l]))
    for (x1, y1, z1, x2, y2, z2) in edges3[l]
        push!(P, to3(x1, y1, z1), to3(x2, y2, z2))
    end
    push!(drawn, linesegments!(ax, P; color = (pal[k], alpha(l)), linewidth = 0.7))
end
box = [Point3f(sx * h, sy * h, sz * h) for (sx, sy, sz) in
       ((-1,-1,-1),(1,-1,-1), (-1,1,-1),(1,1,-1), (-1,-1,1),(1,-1,1), (-1,1,1),(1,1,1),
        (-1,-1,-1),(-1,1,-1), (1,-1,-1),(1,1,-1), (-1,-1,1),(-1,1,1), (1,-1,1),(1,1,1),
        (-1,-1,-1),(-1,-1,1), (1,-1,-1),(1,-1,1), (-1,1,-1),(-1,1,1), (1,1,-1),(1,1,1))]
push!(drawn, linesegments!(ax, box; color = (:white, 0.35), linewidth = 1))
# fixed limits, wide enough for the large cells that reach beyond the cube
R = maximum(maximum(abs, p) for l in used for p in (to3(e[1], e[2], e[3]) for e in edges3[l]))
limits!(ax, -R, R, -R, R, -R, R)
Label(fig3[0, 1:2], "every leaf cell in a $(round(Int, 2h * 1000)) pc cube on a spiral arm, coloured by level";
      color = :white, fontsize = 18)
Colorbar(fig3[1, 2]; colormap = pal, colorrange = LCR, ticks = lticks, label = "level  (cell size)",
         height = Relative(0.55), labelcolor = :white, ticklabelcolor = :white, tickcolor = :white)
save("media/grid_3d.png", fig3)

# The rotation. Turn the cube, not the camera: when the camera turns, Axis3 re-zooms every frame
# to fill the picture, and the cube pulses toward the viewer and back. With the camera and the
# limits fixed, only the cube moves.
# Thin lines compress badly: at the default quality this movie is 35 MB. `compression` is the
# x264 quality setting (higher is smaller); with 32 and 800 pixels it is 2.2 MB.
turn!(φ) = foreach(p -> rotate!(p, Vec3f(0, 0, 1), φ), drawn)
limits!(ax, -√2 * R, √2 * R, -√2 * R, √2 * R, -R, R)   # room for the turned diagonal
resize!(fig3, 800, 800)
t_movie = @elapsed record(turn!, fig3, "media/grid_3d_rotation.mp4",
                          range(0, 2π; length = NFRAMES + 1)[1:end-1];
                          framerate = 30, compression = 32)
resize!(fig3, 400, 400)                          # a small preview that GitHub shows inline
record(turn!, fig3, "media/grid_3d_preview.gif", range(0, 2π; length = 37)[1:end-1]; framerate = 10)
@printf("movie: %d frames in %.0f s\n", NFRAMES, t_movie)
resize!(fig3, 1000, 1000); turn!(0); limits!(ax, -R, R, -R, R, -R, R)
fig3

![Every leaf cell in a cube on a spiral arm](media/grid_3d.png)

![The cube turning](media/grid_3d_preview.gif)

*Reduced preview. Full resolution in [`media/grid_3d_rotation.mp4`](media/grid_3d_rotation.mp4).*

Seen from the side, the finest cells form a layer around the mid-plane, about 100 pc thick. As the
cube turns, the layer turns out to be a ridge that runs diagonally through the cube, along the arm,
wrapped in shells of level-12 and level-11 cells. The large level-10 cells fill the space above
and below it. The rim of large cells that sticks out at the bottom is real: those cells have their
centre inside the cube, so they belong to the selection, but reach beyond it.

## What goes wrong

- **A mean level is not a level.** The projected maps average along the line of sight, so 9.6
  means "mostly level 10, some level 9", not a cell of level 9.6. For whole levels, use the slice.
- **Mass and volume weighting disagree, on purpose.** Mass weighting follows the dense gas, which
  is where the code refines; volume weighting is dominated by the large cells around it. Neither is
  wrong: say which one a figure shows.
- **The edges of a level are a footprint.** Over the full column, `gridoverlay` draws every cell of
  that level anywhere along the line of sight, which looks like refinement everywhere. A slab one
  finest cell thick gives the grid in one plane.
- **Two coordinate frames.** `gridoverlay` returns edges in box coordinates (from the box corner),
  while maps and slices are centred on `center`. Shift the edges by the centre, as above, or the
  grid lands beside the gas. A slice with a window (`xrange`, `yrange`) is measured from the
  middle of that window, not from `center`; for a full, symmetric frame the two agree, for a zoom
  they do not.
- **Too many lines.** Each level has four times as many edges per area as the one below it. Keep
  the window small, or leave out the finest level.
- **A turning view that pulses.** Turning the camera (`azimuth`) makes `Axis3` re-zoom every
  frame to fill the picture, so the cube moves toward the viewer and back, whatever the
  `viewmode`. Turn the drawn objects instead (`rotate!`), with the camera and limits fixed.
- **3-D needs restraint.** Opaque edges turn the cube into a solid block. Draw them transparent,
  the fine levels more transparent than the coarse ones, and keep the cube small: 35 000 cells
  already give over 100 000 edges. Count each shared edge once, or the inner ones are drawn up to
  four times and the cube looks denser than it is.

## The same thing, short

Once the choices are made, the whole recipe is a handful of calls: one table, two projections, one
slice, one overlay.

In [ ]:
lv = getvar(hydro, :level)
counts = [count(==(l), lv) for l in hydro.lmin:hydro.lmax]                 # cells per level

view = (center = CENTER, range_unit = RU, xrange = [-FRAME, FRAME], yrange = [-FRAME, FRAME],
        verbose = false, show_progress = false)
gas_level   = projection(hydro, :level; weighting = [:mass],   view...)    # where the gas sits
space_level = projection(hydro, :level; weighting = [:volume], view...)    # what fills space
plane_level = slice(hydro, :level; los = [0, 0, 1], pxsize = [FRAME / 400, RU],     # exact
                    center = CENTER, range_unit = RU, xrange = [-FRAME, FRAME],
                    yrange = [-FRAME, FRAME], verbose = false)
grid10      = gridoverlay(hydro; level = 10, unit = RU, center = CENTER, range_unit = RU,
                          xrange = collect(ZOOM.x), yrange = collect(ZOOM.y), zrange = slab)
nothing

## The thumbnail

The card on the [gallery page](https://manuelbehrendt.github.io/Mera.jl/stable/gallery/) shows the
3-D view of step 8, which is also the picture at the top of this notebook.

In [ ]:
makethumb("media/grid_3d.png", "media/where_is_the_grid_refined_thumb.png"; mode = :crop)

## Attribution: run this once and paste the result

In [ ]:
using Dates
println("""
| **Reads** | $(info.simcode) |
| **Mera** | $(mera_build()), Julia $(VERSION) |
| **Provenance** | `$(provenance_string(hydro))` |
| **Status** | contributed $(Dates.today()) |
""")

## Making it yours

- **Off-axis**: `gridoverlay` takes the same view keywords as `projection` (`inclination`,
  `azimuth`, `direction = :faceon`), drawing each cell as a square in the camera plane. Pick a
  coarser level there: off-axis edges are not merged, so they are more numerous.
- **Resolution of a structure**: cut a clump or a filament with `subregion` first, then count its
  cells per level as in step 3. That says directly how many cells resolve it.
- **Over time**: the table of step 3 for several outputs shows when and where the code refined.

If you build something with this, a link in
[Discussions](https://github.com/ManuelBehrendt/Mera.jl/discussions) is welcome.